In [14]:
import numpy as np

In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
from pathlib import Path
import pandas as pd
import numpy as np
import cv2
import json
import os
import time

from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download, login

# Project paths
PROJECT_DIR = Path("/content/drive/MyDrive/Nexar_Project")

METADATA_DIR = PROJECT_DIR / "Metadata"
OUTPUT_DIR = PROJECT_DIR / "Dataset_2FPS"

POSITIVE_DIR = OUTPUT_DIR / "positive"
NEGATIVE_DIR = OUTPUT_DIR / "negative"

MANIFEST_DIR = PROJECT_DIR / "Manifests"
LOG_DIR = PROJECT_DIR / "Logs"

for folder in [
    POSITIVE_DIR, NEGATIVE_DIR,
    MANIFEST_DIR, LOG_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

# Sampling settings
FPS_SAMPLE = 2
IMAGE_SIZE = (224, 224)

REPO_ID = "nexar-ai/nexar_collision_prediction"

# Load metadata
metadata_df = pd.read_csv(METADATA_DIR / "all_metadata.csv")

print("Project is ready.")
print("Total videos:", len(metadata_df))
print("Positive:", (metadata_df["label"] == 1).sum())
print("Negative:", (metadata_df["label"] == 0).sum())
print("Sampling rate:", FPS_SAMPLE, "FPS")

Project is ready.
Total videos: 1500
Positive: 750
Negative: 750
Sampling rate: 2 FPS


In [17]:
login()

In [18]:
def process_video(row):

    label = int(row["label"])
    filename = str(row["file_name"])

    category = "positive" if label == 1 else "negative"

    output_dir = POSITIVE_DIR if label == 1 else NEGATIVE_DIR
    output_path = output_dir / f"{Path(filename).stem}.npz"

    # Resume: skip already processed videos
    if output_path.exists():
        return "skipped", 0

    # Download video to Colab's temporary storage
    video_path = hf_hub_download(
        repo_id=REPO_ID,
        filename=f"train/{category}/{filename}",
        repo_type="dataset"
    )

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        raise RuntimeError("Cannot open video")

    video_fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if video_fps <= 0 or total_frames <= 0:
        cap.release()
        raise RuntimeError("Invalid video properties")

    duration = total_frames / video_fps

    # Positive: strictly before collision
    # Negative: entire video
    if label == 1:
        event_time = float(row["time_of_event"])
        end_time = min(event_time, duration)
    else:
        end_time = duration

    frames = []
    timestamps = []

    for timestamp in np.arange(0, end_time, 1 / FPS_SAMPLE):

        cap.set(cv2.CAP_PROP_POS_MSEC, timestamp * 1000)
        success, frame = cap.read()

        if not success:
            continue

        frame = cv2.resize(frame, IMAGE_SIZE)
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        frames.append(frame)
        timestamps.append(timestamp)

    cap.release()

    if not frames:
        raise RuntimeError("No frames extracted")

    # Save locally first, then move complete file to Drive
    temp_path = output_path.with_name(output_path.stem + ".tmp.npz")

    np.savez_compressed(
        temp_path,
        frames=np.asarray(frames, dtype=np.uint8),
        timestamps=np.asarray(timestamps, dtype=np.float32)
    )

    os.replace(temp_path, output_path)

    return "success", len(frames)

In [21]:
import os
import cv2
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download

# ==========================================
# 1. Configuration
# ==========================================

FPS_SAMPLE = 2
IMAGE_SIZE = (224, 224)

SUCCESS_LOG = MANIFEST_DIR / "sampling_success.csv"
ERROR_LOG = MANIFEST_DIR / "sampling_errors.csv"

# ==========================================
# 2. Load Previous Records Safely
# ==========================================

def load_records(path):

    if path.exists() and path.stat().st_size > 0:
        try:
            return pd.read_csv(path).to_dict("records")
        except pd.errors.EmptyDataError:
            return []

    return []


success_records = load_records(SUCCESS_LOG)
error_records = load_records(ERROR_LOG)

print("Previous successful records:", len(success_records))
print("Previous error records:", len(error_records))


# ==========================================
# 3. Check Existing NPZ Files
# ==========================================

def is_valid_npz(path):

    if not path.exists() or path.stat().st_size == 0:
        return False

    try:
        with np.load(path, allow_pickle=False) as data:

            frames = data["frames"]
            timestamps = data["timestamps"]

            return (
                len(frames) > 0
                and len(frames) == len(timestamps)
                and frames.shape[1:] == (224, 224, 3)
            )

    except Exception:
        return False


# ==========================================
# 4. Process One Video
# ==========================================

def process_video(row):

    label = int(row["label"])
    filename = str(row["file_name"])

    category = "positive" if label == 1 else "negative"

    output_dir = POSITIVE_DIR if label == 1 else NEGATIVE_DIR

    output_path = output_dir / f"{Path(filename).stem}.npz"

    # Skip if already processed and valid
    if is_valid_npz(output_path):
        return "skipped", 0

    # Remove incomplete output if present
    if output_path.exists():
        output_path.unlink()

    # Download video from Hugging Face
    video_path = hf_hub_download(
        repo_id=REPO_ID,
        filename=f"train/{category}/{filename}",
        repo_type="dataset"
    )

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        raise RuntimeError("Cannot open video")

    video_fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if video_fps <= 0 or total_frames <= 0:
        cap.release()
        raise RuntimeError("Invalid video properties")

    duration = total_frames / video_fps

    # Positive: only before collision
    # Negative: entire video
    if label == 1:

        event_time = float(row["time_of_event"])

        if not np.isfinite(event_time) or event_time <= 0:
            cap.release()
            raise ValueError("Invalid collision event time")

        end_time = min(event_time, duration)

    else:
        end_time = duration

    frames = []
    timestamps = []

    for timestamp in np.arange(0, end_time, 1 / FPS_SAMPLE):

        cap.set(cv2.CAP_PROP_POS_MSEC, timestamp * 1000)

        success, frame = cap.read()

        if not success:
            continue

        frame = cv2.resize(frame, IMAGE_SIZE)

        frame = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        frames.append(frame)
        timestamps.append(timestamp)

    cap.release()

    if not frames:
        raise RuntimeError("No frames extracted")

    # Save to temporary file first
    temp_path = output_dir / f"{Path(filename).stem}.tmp.npz"

    np.savez_compressed(
        temp_path,
        frames=np.asarray(frames, dtype=np.uint8),
        timestamps=np.asarray(timestamps, dtype=np.float32)
    )

    # Move completed file to final destination
    os.replace(temp_path, output_path)

    return "success", len(frames)


# ==========================================
# 5. Resume Full Dataset Processing
# ==========================================

error_records = []
success_records = []

for _, row in tqdm(
    metadata_df.iterrows(),
    total=len(metadata_df),
    desc="Processing videos"
):

    filename = str(row["file_name"])
    label = int(row["label"])

    try:

        status, frame_count = process_video(row)

        success_records.append({
            "file_name": filename,
            "label": label,
            "frames": frame_count,
            "status": status
        })

    except Exception as e:

        error_records.append({
            "file_name": filename,
            "label": label,
            "error": str(e)
        })

    # Save checkpoint after every 10 videos
    if (len(success_records) + len(error_records)) % 10 == 0:

        pd.DataFrame(
            success_records,
            columns=["file_name", "label", "frames", "status"]
        ).to_csv(SUCCESS_LOG, index=False)

        pd.DataFrame(
            error_records,
            columns=["file_name", "label", "error"]
        ).to_csv(ERROR_LOG, index=False)


# ==========================================
# 6. Save Final Reports
# ==========================================

pd.DataFrame(
    success_records,
    columns=["file_name", "label", "frames", "status"]
).to_csv(SUCCESS_LOG, index=False)

pd.DataFrame(
    error_records,
    columns=["file_name", "label", "error"]
).to_csv(ERROR_LOG, index=False)


# ==========================================
# 7. Final Summary
# ==========================================

positive_count = sum(
    1 for row in metadata_df[metadata_df["label"] == 1].itertuples()
    if is_valid_npz(POSITIVE_DIR / f"{Path(row.file_name).stem}.npz")
)

negative_count = sum(
    1 for row in metadata_df[metadata_df["label"] == 0].itertuples()
    if is_valid_npz(NEGATIVE_DIR / f"{Path(row.file_name).stem}.npz")
)

print("\nSampling process completed.")
print("Positive videos:", positive_count)
print("Negative videos:", negative_count)
print("Total processed:", positive_count + negative_count)
print("Remaining:", len(metadata_df) - positive_count - negative_count)
print("Errors in this run:", len(error_records))

Previous successful records: 50
Previous error records: 0


Processing videos:   0%|          | 0/1500 [00:00<?, ?it/s]

train/positive/00148.mp4: reconstructing file:   0%|          |  0.00B / 17.4MB            

train/positive/00148.mp4: downloading bytes:           |  0.00B            

train/positive/00301.mp4: reconstructing file:   0%|          |  0.00B / 14.0MB            

train/positive/00301.mp4: downloading bytes:           |  0.00B            

... (349 outputs left out)

train/positive/00856.mp4: downloading bytes:           |  0.00B            

train/positive/00607.mp4: reconstructing file:   0%|          |  0.00B / 21.4MB            

train/positive/00607.mp4: downloading bytes:           |  0.00B            

KeyboardInterrupt: 

In [ ]:

import os
import cv2
import time
import shutil
import numpy as np
import pandas as pd

from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

# ==========================================
# 1. Configuration
# ==========================================

PROJECT_DIR = Path("/content/drive/MyDrive/Nexar_Project")

METADATA_PATH = PROJECT_DIR / "Metadata" / "all_metadata.csv"

DRIVE_OUTPUT = PROJECT_DIR / "Dataset_2FPS"
DRIVE_POSITIVE = DRIVE_OUTPUT / "positive"
DRIVE_NEGATIVE = DRIVE_OUTPUT / "negative"

LOG_DIR = PROJECT_DIR / "Logs"

LOCAL_DIR = Path("/content/Nexar_Staging")
LOCAL_POSITIVE = LOCAL_DIR / "positive"
LOCAL_NEGATIVE = LOCAL_DIR / "negative"

REPO_ID = "nexar-ai/nexar_collision_prediction"

FPS_SAMPLE = 2
IMAGE_SIZE = (224, 224)

MAX_WORKERS = 3
BATCH_SIZE = 20

# ==========================================
# 2. Create directories
# ==========================================

for folder in [
    DRIVE_POSITIVE,
    DRIVE_NEGATIVE,
    LOG_DIR,
    LOCAL_POSITIVE,
    LOCAL_NEGATIVE
]:
    folder.mkdir(parents=True, exist_ok=True)

# ==========================================
# 3. Load metadata
# ==========================================

df = pd.read_csv(METADATA_PATH)

df["label"] = pd.to_numeric(df["label"], errors="raise")

if not set(df["label"].dropna().unique()).issubset({0, 1}):
    raise ValueError("Unexpected label values")

print("Total videos:", len(df))
print("Positive videos:", (df["label"] == 1).sum())
print("Negative videos:", (df["label"] == 0).sum())

# ==========================================
# 4. Validate NPZ
# ==========================================

def is_valid_npz(path):

    if not path.exists() or path.stat().st_size == 0:
        return False

    try:
        with np.load(path) as data:

            frames = data["frames"]
            timestamps = data["timestamps"]

            return (
                len(frames) > 0
                and len(frames) == len(timestamps)
                and frames.shape[1:] == (224, 224, 3)
            )

    except Exception:
        return False

# ==========================================
# 5. Process one video
# ==========================================

def process_video(row):

    filename = str(row["file_name"])
    label = int(row["label"])

    is_positive = label == 1
    category = "positive" if is_positive else "negative"

    local_dir = LOCAL_POSITIVE if is_positive else LOCAL_NEGATIVE
    local_output = local_dir / f"{Path(filename).stem}.npz"

    if is_valid_npz(local_output):
        return {
            "status": "success",
            "file_name": filename,
            "label": label,
            "local_path": str(local_output),
            "frames": None
        }

    hf_path = f"train/{category}/{filename}"

    video_path = hf_hub_download(
        repo_id=REPO_ID,
        filename=hf_path,
        repo_type="dataset"
    )

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        raise RuntimeError("Could not open video")

    try:

        fps = cap.get(cv2.CAP_PROP_FPS)
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        if fps <= 0 or frame_count <= 0:
            raise RuntimeError("Invalid video properties")

        duration = frame_count / fps

        if is_positive:

            event_time = float(row["time_of_event"])

            if not np.isfinite(event_time) or event_time <= 0:
                raise ValueError("Invalid event time")

            end_time = min(event_time, duration)

        else:
            end_time = duration

        timestamps = np.arange(
            0,
            end_time,
            1 / FPS_SAMPLE
        )

        frame_indices = np.unique(
            np.round(timestamps * fps).astype(int)
        )

        frames = []
        saved_timestamps = []

        current_frame = 0

        for target_idx in frame_indices:

            while current_frame <= target_idx:

                if not cap.grab():
                    break

                current_frame += 1

            if current_frame <= target_idx:
                break

            success, frame = cap.retrieve()

            if not success:
                continue

            frame = cv2.resize(frame, IMAGE_SIZE)
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            frames.append(frame)
            saved_timestamps.append(target_idx / fps)

        if not frames:
            raise RuntimeError("No frames extracted")

        temp_path = local_output.with_name(
            local_output.stem + ".tmp.npz"
        )

        np.savez_compressed(
            temp_path,
            frames=np.asarray(frames, dtype=np.uint8),
            timestamps=np.asarray(saved_timestamps, dtype=np.float32)
        )

        os.replace(temp_path, local_output)

        return {
            "status": "success",
            "file_name": filename,
            "label": label,
            "local_path": str(local_output),
            "frames": len(frames)
        }

    finally:
        cap.release()

# ==========================================
# 6. Find remaining videos
# ==========================================

jobs = []

for _, row in df.iterrows():

    label = int(row["label"])
    filename = str(row["file_name"])

    drive_dir = DRIVE_POSITIVE if label == 1 else DRIVE_NEGATIVE

    drive_output = drive_dir / f"{Path(filename).stem}.npz"

    if not is_valid_npz(drive_output):
        jobs.append(row.to_dict())

print("\nVideos remaining:", len(jobs))

# ==========================================
# 7. Transfer files to Drive
# ==========================================

def transfer_to_drive(result):

    local_path = Path(result["local_path"])
    label = int(result["label"])

    drive_dir = DRIVE_POSITIVE if label == 1 else DRIVE_NEGATIVE
    drive_path = drive_dir / local_path.name

    if is_valid_npz(drive_path):
        return "already_exists"

    if not is_valid_npz(local_path):
        raise RuntimeError("Local NPZ validation failed")

    temp_path = drive_path.with_name(
        drive_path.stem + ".uploading.npz"
    )

    shutil.copy2(local_path, temp_path)

    if not is_valid_npz(temp_path):
        temp_path.unlink(missing_ok=True)
        raise RuntimeError("Drive copy validation failed")

    os.replace(temp_path, drive_path)

    return "transferred"

# ==========================================
# 8. Parallel processing
# ==========================================

success_records = []
error_records = []
pending_transfers = []

start_time = time.time()

def save_logs():

    pd.DataFrame(success_records).to_csv(
        LOG_DIR / "sampling_success.csv",
        index=False
    )

    pd.DataFrame(error_records).to_csv(
        LOG_DIR / "sampling_errors.csv",
        index=False
    )

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:

    futures = {
        executor.submit(process_video, row): row
        for row in jobs
    }

    for future in tqdm(
        as_completed(futures),
        total=len(futures),
        desc="Processing videos"
    ):

        try:

            result = future.result()
            pending_transfers.append(result)

        except Exception as error:

            row = futures[future]

            error_records.append({
                "file_name": row["file_name"],
                "error": str(error)
            })

        if len(pending_transfers) >= BATCH_SIZE:

            for item in pending_transfers:

                try:

                    transfer_to_drive(item)

                    success_records.append({
                        "file_name": item["file_name"],
                        "label": item["label"],
                        "frames": item["frames"],
                        "status": "saved"
                    })

                except Exception as error:

                    error_records.append({
                        "file_name": item["file_name"],
                        "error": str(error)
                    })

            pending_transfers.clear()
            save_logs()

# ==========================================
# 9. Transfer remaining files
# ==========================================

for item in pending_transfers:

    try:

        transfer_to_drive(item)

        success_records.append({
            "file_name": item["file_name"],
            "label": item["label"],
            "frames": item["frames"],
            "status": "saved"
        })

    except Exception as error:

        error_records.append({
            "file_name": item["file_name"],
            "error": str(error)
        })

save_logs()

# ==========================================
# 10. Final report
# ==========================================

positive_count = sum(
    is_valid_npz(p) for p in DRIVE_POSITIVE.glob("*.npz")
)

negative_count = sum(
    is_valid_npz(p) for p in DRIVE_NEGATIVE.glob("*.npz")
)

elapsed = (time.time() - start_time) / 60

print("\nProcessing completed.")
print("Newly saved videos:", len(success_records))
print("Failed videos:", len(error_records))
print("Positive videos in Drive:", positive_count)
print("Negative videos in Drive:", negative_count)
print("Total valid videos:", positive_count + negative_count)
print("Elapsed time (minutes):", round(elapsed, 2))


Total videos: 1500
Positive videos: 750
Negative videos: 750

Videos remaining: 1265


Processing videos:   0%|          | 0/1265 [00:00<?, ?it/s]

train/positive/00947.mp4: reconstructing file:   0%|          |  0.00B / 16.9MB            

train/positive/00902.mp4: reconstructing file:   0%|          |  0.00B / 19.6MB            

train/positive/00947.mp4: downloading bytes:           |  0.00B            

train/positive/00902.mp4: downloading bytes:           |  0.00B            

... (754 outputs left out)

train/positive/00996.mp4: reconstructing file:   0%|          |  0.00B / 32.1MB            

train/positive/00996.mp4: downloading bytes:           |  0.00B            

train/positive/00912.mp4: reconstructing file:   0%|          |  0.00B / 32.5MB            

train/positive/00912.mp4: downloading bytes:           |  0.00B            

In [25]:

import pandas as pd

metadata_path = "/content/drive/MyDrive/Nexar_Project/Metadata/all_metadata.csv"

df = pd.read_csv(metadata_path)

print("Columns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nData types:")
print(df.dtypes)

print("\nExisting NPZ files:")
import os

positive_dir = "/content/drive/MyDrive/Nexar_Project/Dataset_2FPS/positive"
negative_dir = "/content/drive/MyDrive/Nexar_Project/Dataset_2FPS/negative"

print("Positive:", len(os.listdir(positive_dir)))
print("Negative:", len(os.listdir(negative_dir)))

Columns:
['file_name', 'time_of_event', 'time_of_alert', 'light_conditions', 'weather', 'scene', 'time_to_accident', 'label']

First 5 rows:


,file_name,time_of_event,time_of_alert,light_conditions,weather,scene,time_to_accident,label
0,00822.mp4,19.500,18.633,Normal,Cloudy,Urban,NaN,1
1,00208.mp4,19.800,19.233,Normal,Cloudy,Sub-urban,NaN,1
2,00072.mp4,20.101,20.068,Normal,Clear,Urban,NaN,1
3,00128.mp4,19.267,18.333,Normal,Clear,Urban,NaN,1
4,00205.mp4,21.146,18.196,Normal,Clear,Highway,NaN,1



Data types:
file_name            object
time_of_event       float64
time_of_alert       float64
light_conditions     object
weather              object
scene                object
time_to_accident    float64
label                 int64
dtype: object

Existing NPZ files:
Positive: 235
Negative: 0
